In [52]:
import csv
import json
import pandas as pd
import numpy as np
from pydantic import ValidationError,field_validator
from datetime import datetime

In [53]:
import os

os.makedirs("data", exist_ok=True)

In [54]:
def parse_row(row):
    parsed = row.copy()
    Schema = {
        "age":int,
        "fee_inr":int,
        "prior_no_shows":int,
        "distance_km":float
    }
    
    for column,data_type in Schema.items():
        try:
            parsed[column] = data_type(parsed[column])
        except ValueError as e:
            return None,column,e

    return parsed,None,None

    

In [55]:
with open("data/appointment.csv", newline="") as appointment_csv:
    reader = csv.DictReader(appointment_csv)
    headers = reader.fieldnames    
    print(f"Header: {headers}")
    rejected_rows = []
    active_rows = []
    for row in reader:
        result,column,err = parse_row(row)       
        if result is None:            
            #rejected_rows.append({"row":row,"failed_column":column,"Error":err})
            rejected_rows.append(row)
        else:
            active_rows.append(result)

    if headers and "appointment_id" in headers:
        print(f"Total Rows:{reader.line_num-1}")
    else:
        print(f"Total Rows:{reader.line_num}")

    print(len(rejected_rows))


Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total Rows:1020
32


In [56]:
with open("data/rejected_rows.json", "w", encoding="utf-8") as f:
  json.dump(rejected_rows, f, indent=2)

In [57]:
df = pd.read_csv("data/appointment.csv")


In [58]:
df.shape

(1020, 14)

In [59]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   str    
 1   patient_id       1020 non-null   str    
 2   clinic_id        1020 non-null   str    
 3   booked_on        1020 non-null   str    
 4   appt_date        1019 non-null   str    
 5   age              1008 non-null   float64
 6   gender           1020 non-null   str    
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   str    
 10  visit_type       1020 non-null   str    
 11  reminder_sent    1020 non-null   str    
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   str    
dtypes: float64(2), int64(2), str(10)
memory usage: 111.7 KB


In [60]:
print("Null counts:\n", df.isnull().sum())

Null counts:
 appointment_id      0
patient_id          0
clinic_id           0
booked_on           0
appt_date           1
age                12
gender              0
distance_km        20
fee_inr             0
booking_channel     0
visit_type          0
reminder_sent       0
prior_no_shows      0
status              4
dtype: int64


In [61]:
print("Duplicate count:", df.duplicated().sum())

Duplicate count: 20


In [62]:
df = df.drop_duplicates()

In [63]:
print("After dropping duplicates:", len(df))

After dropping duplicates: 1000


In [64]:
df["booking_channel"] = df["booking_channel"].str.strip().str.title()

In [ ]:
df['appt_date'] = pd.to_datetime(df["appt_date"],errors="coerce")
df['booked_on'] = pd.to_datetime(df["booked_on"],errors="coerce")

ValueError: errors must be one of 'raise', or 'coerce'.

In [ ]:
df = df.dropna(subset=["appt_date", "status"])
print("After dropping missing dates/status:", len(df))